In [1]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [2]:
@dataclass
class Universe:
  Bonds:List[str]
  ManagedFutures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

@dataclass
class HERCParams:
  quantile:float|None=0.95
  use_lw_shrinkage:bool=False
  k_max:int=5
  n_sims:int=100


In [197]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark


  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [198]:
class Optimizer:
  def __init__(self, optim_params, debug: bool = False):
    self.p = optim_params
    self.debug = debug

  def get_asset_w(self, returns: pd.DataFrame):
    T, N = returns.shape
    R = returns.values

    if N == 1:
      col = returns.columns[0]
      r = R.flatten()

      var = np.quantile(-r, self.p.quantile)
      tail = -r[-r >= var]

      cvar = tail.mean() if len(tail) else var
      cvar = max(cvar, 1e-6)

      return pd.Series([1.0], index=[col]), cvar

    y = cp.Variable(N, nonneg=True, name="raw_weights")
    u = cp.Variable(T, nonneg=True, name="slack_losses")
    b = np.ones(N) / N
    zeta = cp.Variable(name="VaR")

    CVaR = zeta + (1 / ((1 - self.p.quantile) * T)) * cp.sum(u)
    log_barrier = cp.sum(cp.multiply(b, cp.log(y)))

    eps = 1e-3
    reg = eps * cp.sum(y)

    constraints = [
        u >= -R @ y - zeta,
        u >= 0,
        y >= 1e-4,
        y <= 1e+3
    ]

    obj_fn = cp.Minimize(CVaR + reg - log_barrier)
    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL, verbose=self.debug)

    if problem.status not in ("optimal", "optimal_inaccurate"):
      raise RuntimeError(f"Solver failed for cluster {list(returns.columns)}: {problem.status}")

    w_raw = y.value
    w = w_raw / w_raw.sum()
    w = pd.Series(w, index=returns.columns)

    cvar_c = CVaR.value / w_raw.sum()

    return w, cvar_c

  def get_cluster_w(self, risk_measure: np.ndarray, eps: float = 1e-6):
    risk_measure = np.maximum(np.asarray(risk_measure), eps)
    inv_risk = 1 / risk_measure

    return inv_risk / inv_risk.sum()

In [200]:
class HERCOptimizer(Optimizer):
  def __init__(self, optim_params: HERCParams, debug=False, **kwargs):
    super().__init__(optim_params=optim_params, debug=debug, **kwargs)

  def _compute_cl_dispersion(self, dist_mtx, cluster_labels):
    unique_clusters = np.unique(cluster_labels)
    W_k = 0.0
    for c_id in unique_clusters:
      cluster_indices = np.where(cluster_labels == c_id)[0]
      cluster_dist = dist_mtx[np.ix_(cluster_indices, cluster_indices)]

      norm = 2.0 * len(cluster_indices)

      D_r = np.sum(cluster_dist ** 2)

      W_k += D_r / norm

    return W_k

  def _generate_null_reference(self, returns_df):
    N_samples, N_assets = returns_df.shape

    min_bounds = returns_df.min(axis=0).values
    max_bounds = returns_df.max(axis=0).values

    null_dists, null_linkages = [], []

    for _ in range(self.p.n_sims):
      null_returns = np.random.uniform(low=min_bounds, high=max_bounds, size=(N_samples, N_assets))
      null_corr = np.clip(np.corrcoef(null_returns, rowvar=False), -1.0, 1.0)
      null_dist = np.sqrt(2.0 * (1.0 - null_corr))

      condensed_dist = squareform(null_dist, checks=False)

      Z_null = linkage(condensed_dist, method="single")

      null_dists.append(null_dist)
      null_linkages.append(Z_null)
    return null_dists, null_linkages

  def _compute_ref_log_disp(self, null_dists, null_linkages, k):
    B = self.p.n_sims
    W_k_log = []

    for b in range(B):
      clusters = fcluster(null_linkages[b], t=k, criterion="maxclust")

      W_k = self._compute_cl_dispersion(null_dists[b], clusters)
      W_k_log.append(np.log(max(W_k, 1e-300)))

    W_k_log = np.array(W_k_log)
    E_W_k = np.mean(W_k_log)

    sdk = np.std(W_k_log, ddof=1) if B > 1 else 0.0
    s_k = sdk * np.sqrt(1.0 + 1.0 / B)

    return E_W_k, s_k

  def _get_k_clusters(self, dist_matrix, Z, returns_df):
    k_max = min(self.p.k_max, dist_matrix.shape[0] - 1)
    Gap_k, s_k_list = [], []
    null_dists, null_linkages = self._generate_null_reference(returns_df)
    k_range = list(range(1, k_max + 1))

    for k in k_range:
      clusters = fcluster(Z, t=k, criterion="maxclust")

      W_k_real = self._compute_cl_dispersion(dist_matrix, clusters)
      log_disp_k = np.log(max(W_k_real, 1e-300))

      E_W_k, s_k = self._compute_ref_log_disp(null_dists, null_linkages, k)

      Gap_k.append(E_W_k - log_disp_k)
      s_k_list.append(s_k)

    gaps = np.array(Gap_k)
    s_k_list = np.array(s_k_list)

    optimal_k = k_range[np.argmax(gaps)]
    for idx in range(len(k_range) - 1):
      if gaps[idx] >= gaps[idx + 1] - s_k_list[idx + 1]:
        optimal_k = k_range[idx]
        break

    optimal_k = max(optimal_k, 2) if dist_matrix.shape[0] > 1 else optimal_k
    return int(optimal_k)

  def get_clusters(self, returns: pd.DataFrame) -> pd.DataFrame:
    corr = returns.corr().values
    dist_mtx = np.sqrt(2.0 * (1.0 - corr))

    comp_disp = squareform(dist_mtx, checks=False)
    Z = linkage(comp_disp, method="single")

    optimal_k = self._get_k_clusters(dist_mtx, Z, returns)
    labels = fcluster(Z, t=optimal_k, criterion="maxclust")

    return pd.DataFrame({'Asset': returns.columns, 'Cluster': labels}), Z

  def _ordered_cluster_ids(self, Z, clusters_df):
    label_by_asset = clusters_df.set_index('Asset')['Cluster']
    leaf_order = leaves_list(Z)

    asset_order = clusters_df['Asset'].values[leaf_order]
    ordered_labels = label_by_asset.loc[asset_order].values

    seen, ordered_ids = set(), []
    for lbl in ordered_labels:
      if lbl not in seen:
        seen.add(lbl)
        ordered_ids.append(lbl)

    return ordered_ids

  def _recursive_bisect(self, ordered_ids, cluster_return_series):
    weights = {cid: 1.0 for cid in ordered_ids}

    def risk_of(ids):
      if len(ids) == 1:
        return self.get_asset_w.__self__ and None

      combo = np.mean([cluster_return_series[c] for c in ids], axis=0)

      var = np.quantile(-combo, self.p.quantile)
      tail = -combo[-combo >= var]
      cvar = tail.mean() if len(tail) else var
      return max(cvar, 1e-6)

    def node_risk(ids):
      if len(ids) == 1:
        r = cluster_return_series[ids[0]]
        var = np.quantile(-r, self.p.quantile)
        tail = -r[-r >= var]
        cvar = tail.mean() if len(tail) else var

        return max(cvar, 1e-6)

      combo = np.mean([cluster_return_series[c] for c in ids], axis=0)
      var = np.quantile(-combo, self.p.quantile)
      tail = -combo[-combo >= var]
      cvar = tail.mean() if len(tail) else var

      return max(cvar, 1e-6)

    def bisect(ids):
      if len(ids) <= 1:
        return

      mid = len(ids) // 2
      left, right = ids[:mid], ids[mid:]
      r_l, r_r = node_risk(left), node_risk(right)
      alpha = 1 - r_l / (r_l + r_r)

      for c in left:  weights[c] *= alpha
      for c in right: weights[c] *= (1 - alpha)

      bisect(left)
      bisect(right)

    bisect(ordered_ids)
    return weights

  def optimize_w(self, returns: pd.DataFrame):
    clusters_df, Z = self.get_clusters(returns)
    unique = np.unique(clusters_df.Cluster.values)

    w_by_cluster = {}
    cluster_return_series = {}

    for c_id in unique:
      cols_i = clusters_df.loc[clusters_df.Cluster == c_id, 'Asset'].values
      returns_i = returns[cols_i]

      w_i, cvar_i = self.get_asset_w(returns_i)
      w_by_cluster[c_id] = w_i

      cluster_return_series[c_id] = (returns_i.values @ w_i.values)

    ordered_ids = self._ordered_cluster_ids(Z, clusters_df)
    cluster_weights = self._recursive_bisect(ordered_ids, cluster_return_series)

    w_final = []
    for c_id, w_i in w_by_cluster.items():
      w_final.append(w_i * cluster_weights[c_id])

    w_df = pd.concat(w_final)
    return w_df

In [201]:
class Portfolio(DataStore, HERCOptimizer):
  def __init__(self, optim_params, debug: bool = False, **kwargs):
    super().__init__(debug=debug, optim_params=optim_params, **kwargs)

  def get_data(self, universe, start, end):
    data, benchmark = self._get_data(universe=universe, start=start, end=end)
    returns = data.pct_change().dropna()
    return returns, benchmark

  def optimize(self, returns, plot=True):
    w = self.optimize_w(returns)
    if plot:
      w.plot.bar()
      plt.show()
    return w

In [202]:
@dataclass
class BacktestParams:
  lookback:int = 252*2
  rebalance_freq:int = 21
  window_type:str = "rolling"
  min_train:int = 252
  n_bootstrap:int = 1000
  block_size:Optional[int] = None
  bootstrap_method:str = "stationary"
  ci_alpha:float = 0.05
  periods_per_year:int = 252
  rf:float = 0.0

  leverage_mode:str = "none"
  leverage:float = 1.0
  target_vol:float = 0.10
  vol_lookback:int = 63
  min_leverage:float = 1.0
  max_leverage:float = 3.0
  borrow_rate:float = 0.0

In [203]:
class Backtest:
  def __init__(self, portfolio, bt_params: BacktestParams, debug: bool = False):
    self.ptf = portfolio
    self.p = bt_params
    self.debug = debug

    self.oos_returns_: Optional[pd.Series] = None
    self.unlevered_returns_: Optional[pd.Series] = None
    self.leverage_history_: Optional[pd.Series] = None
    self.weights_history_: Optional[pd.DataFrame] = None
    self.failed_windows_: List[pd.Timestamp] = []

  def _compute_leverage(self, train_returns: pd.DataFrame, w: pd.Series) -> float:
    if self.p.leverage_mode == "none":
      return 1.0

    if self.p.leverage_mode == "fixed":
      return float(np.clip(self.p.leverage, self.p.min_leverage, self.p.max_leverage))

    if self.p.leverage_mode == "vol_target":
      lb = min(self.p.vol_lookback, len(train_returns))
      recent = train_returns.iloc[-lb:]
      w_aligned = w.reindex(recent.columns).fillna(0.0)
      port_ret = recent @ w_aligned

      realized_vol = port_ret.std(ddof=1) * np.sqrt(self.p.periods_per_year)
      if not realized_vol or np.isnan(realized_vol):
        return 1.0

      lev = self.p.target_vol / realized_vol
      return float(np.clip(lev, self.p.min_leverage, self.p.max_leverage))

    raise ValueError(f"Unknown leverage_mode: {self.p.leverage_mode}")

  def _get_windows(self, returns: pd.DataFrame):
    T = len(returns)
    windows = []
    train_end = self.p.min_train

    while train_end < T:
      test_start = train_end
      test_end = min(test_start + self.p.rebalance_freq, T)

      if self.p.window_type == "rolling":
        train_start = max(0, train_end - self.p.lookback)
      else:
        train_start = 0

      windows.append((train_start, train_end, test_start, test_end))
      train_end = test_end

    return windows

  def run(self, returns: pd.DataFrame) -> pd.Series:
    windows = self._get_windows(returns)
    oos_chunks, unlevered_chunks = [], []
    weight_records, leverage_records = [], []
    self.failed_windows_ = []

    for tr_s, tr_e, te_s, te_e in windows:
      train_returns = returns.iloc[tr_s:tr_e]
      test_returns = returns.iloc[te_s:te_e]
      rebal_date = returns.index[te_s]

      try:
        w = self.ptf.optimize_w(train_returns)
      except Exception as e:
        if self.debug:
          print(f"[{rebal_date.date()}] optimization failed ({e}); falling back to equal weight")
        self.failed_windows_.append(rebal_date)
        w = pd.Series(1.0 / train_returns.shape[1], index=train_returns.columns)

      w = w.reindex(test_returns.columns).fillna(0.0)
      lev = self._compute_leverage(train_returns, w)

      port_ret_unlevered = test_returns @ w
      borrow_cost = max(lev - 1.0, 0.0) * self.p.borrow_rate / self.p.periods_per_year
      port_ret_levered = lev * port_ret_unlevered - borrow_cost

      unlevered_chunks.append(port_ret_unlevered)
      oos_chunks.append(port_ret_levered)
      weight_records.append(w.rename(rebal_date))
      leverage_records.append(pd.Series(lev, index=test_returns.index))

      if self.debug:
        print(f"Rebalanced {rebal_date.date()} | train {returns.index[tr_s].date()}–{returns.index[tr_e-1].date()} "
              f"({tr_e - tr_s} bars) | lev={lev:.2f} | test {test_returns.index[0].date()}–{test_returns.index[-1].date()}")

    self.unlevered_returns_ = pd.concat(unlevered_chunks).sort_index()
    self.unlevered_returns_.name = "strategy_unlevered"

    self.oos_returns_ = pd.concat(oos_chunks).sort_index()
    self.oos_returns_.name = "strategy"

    self.leverage_history_ = pd.concat(leverage_records).sort_index()
    self.leverage_history_.name = "leverage"

    self.weights_history_ = pd.DataFrame(weight_records)

    return self.oos_returns_

  def bootstrap_metrics(self, quantile: float = 0.95, seed: Optional[int] = None) -> pd.DataFrame:
    if self.oos_returns_ is None:
      raise RuntimeError("Call .run(returns) before .bootstrap_metrics().")

    r = self.oos_returns_.values
    T = len(r)
    block_size = self.p.block_size or max(int(np.sqrt(T)), 2)
    rng = np.random.default_rng(seed)

    point_est = self._compute_metrics(r, quantile)
    boot = {k: np.empty(self.p.n_bootstrap) for k in point_est}

    for i in range(self.p.n_bootstrap):
      idx = self._resample_indices(T, block_size, rng)
      m = self._compute_metrics(r[idx], quantile)
      for k, v in m.items():
        boot[k][i] = v

    lo_q, hi_q = self.p.ci_alpha / 2, 1 - self.p.ci_alpha / 2
    summary = {}
    for k, vals in boot.items():
      vals = vals[~np.isnan(vals)]
      summary[k] = {
        "estimate": point_est[k],
        "ci_lower": np.quantile(vals, lo_q) if len(vals) else np.nan,
        "ci_upper": np.quantile(vals, hi_q) if len(vals) else np.nan,
        "std_err": vals.std(ddof=1) if len(vals) > 1 else np.nan,
      }

    return pd.DataFrame(summary).T

  def _compute_metrics(self, arr: np.ndarray, quantile: float) -> Dict[str, float]:
    r = pd.Series(arr)
    return {
      "CAGR": self._cagr(r, self.p.periods_per_year),
      "Vol": self._vol(r, self.p.periods_per_year),
      "Sharpe": self._sharpe(r, self.p.periods_per_year, self.p.rf),
      "Sortino": self._sortino(r, self.p.periods_per_year, self.p.rf),
      "MaxDD": self._max_dd(r),
      "Calmar": self._calmar(r, self.p.periods_per_year),
      "CVaR": self._cvar(r, quantile),
    }

  @staticmethod
  def _cagr(r: pd.Series, ppy: int) -> float:
    n_years = len(r) / ppy
    if n_years <= 0:
      return np.nan
    cum = (1 + r).prod()
    return cum ** (1 / n_years) - 1

  @staticmethod
  def _vol(r: pd.Series, ppy: int) -> float:
    return r.std(ddof=1) * np.sqrt(ppy)

  @staticmethod
  def _sharpe(r: pd.Series, ppy: int, rf: float) -> float:
    excess = r - rf / ppy
    sigma = r.std(ddof=1)
    if not sigma or np.isnan(sigma):
      return np.nan
    return (excess.mean() / sigma) * np.sqrt(ppy)

  @staticmethod
  def _sortino(r: pd.Series, ppy: int, rf: float) -> float:
    excess = r - rf / ppy
    downside = r[r < 0]
    dd = downside.std(ddof=1)
    if not dd or np.isnan(dd):
      return np.nan
    return (excess.mean() / dd) * np.sqrt(ppy)

  @staticmethod
  def _max_dd(r: pd.Series) -> float:
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    return (cum / peak - 1).min()

  @classmethod
  def _calmar(cls, r: pd.Series, ppy: int) -> float:
    cagr = cls._cagr(r, ppy)
    mdd = cls._max_dd(r)
    if not mdd or np.isnan(mdd):
      return np.nan
    return cagr / abs(mdd)

  @staticmethod
  def _cvar(r: pd.Series, quantile: float) -> float:
    x = r.values if isinstance(r, pd.Series) else np.asarray(r)
    var = np.quantile(-x, quantile)
    tail = -x[-x >= var]
    return tail.mean() if len(tail) else var

  def _resample_indices(self, T: int, block_size: int, rng: np.random.Generator) -> np.ndarray:
    if self.p.bootstrap_method == "stationary":
      return self._stationary_bootstrap_indices(T, block_size, rng)
    elif self.p.bootstrap_method == "circular":
      return self._circular_block_bootstrap_indices(T, block_size, rng)
    else:
      raise ValueError(f"Unknown bootstrap_method: {self.p.bootstrap_method}")

  @staticmethod
  def _stationary_bootstrap_indices(T: int, block_size: int, rng: np.random.Generator) -> np.ndarray:
    p = 1.0 / block_size
    indices = np.empty(T, dtype=int)
    idx = rng.integers(0, T)

    for t in range(T):
      indices[t] = idx
      idx = rng.integers(0, T) if rng.random() < p else (idx + 1) % T

    return indices

  @staticmethod
  def _circular_block_bootstrap_indices(T: int, block_size: int, rng: np.random.Generator) -> np.ndarray:
    n_blocks = int(np.ceil(T / block_size))
    starts = rng.integers(0, T, size=n_blocks)
    idx = np.concatenate([(np.arange(block_size) + s) % T for s in starts])
    return idx[:T]

  def plot_oos(self, benchmark: Optional[pd.Series] = None):
    cum_strat = (1 + self.oos_returns_).cumprod()
    fig, ax = plt.subplots(figsize=(15, 8))
    cum_strat.plot(ax=ax, label="Strategy (OOS)")

    if benchmark is not None:
      bench_aligned = benchmark.reindex(self.oos_returns_.index).fillna(0.0)
      (1 + bench_aligned).cumprod().plot(ax=ax, label="Benchmark")

    ax.legend()
    ax.set_title("Walk-Forward Out-of-Sample Performance")
    plt.show()

  def plot_leverage(self):
    fig, ax = plt.subplots(figsize=(15, 4))
    self.leverage_history_.plot(ax=ax)
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=1)
    ax.set_title("Leverage Over Time")
    ax.set_ylabel("Multiplier")
    plt.show()